## XBRL US API - Python example  
This sample Python code queries assertion data for SEC filings.

### Authenticate for access token 
Click the run button and enter your XBRL US Web account email, account password, Client ID, and secret (get these from https://xbrl.us/access-token), pressing the Enter key on the keyboard after each entry.

XBRL US limits records returned for a query to improve efficiency; this script loops to collect all data from the Public Filings Database for a query. **Non-members might not be able to return all data for a query** - join XBRL US for comprehensive access - https://xbrl.us/join.

In [ ]:
# @title
import os, re, sys, json
import requests
import pandas as pd
from IPython.display import display, HTML
import numpy as np
import getpass
from datetime import datetime
import urllib
from urllib.parse import urlencode
api = input('Enter subdomain ("api" or left blank to query public results, otherwise enter a value) ') or 'api'
baseurl = 'https://' + api + '.xbrl.us/'

class tokenInfoClass:
    access_token = None
    refresh_token = None
    email = None
    username = None
    client_id = None
    client_secret = None
    authurl = baseurl + 'oauth2/token'
    headers = {"Content-Type": "application/x-www-form-urlencoded"}
	
def refresh(info):
    refresh_auth = {
                'client_id': info.client_id, 
				'client_secret' : info.client_secret, 
				'grant_type' : 'refresh_token', 
				'platform' : 'ipynb', 
				'refresh_token' : info.refresh_token 
                }
    refreshres = requests.post(info.authurl, data=refresh_auth, headers=info.headers)
    refresh_json = refreshres.json()
    info.access_token = refresh_json['access_token']
    info.refresh_token = refresh_json['refresh_token']
    print('Your access token(%s) is refreshed for 60 minutes. If it expires again, run this cell to generate a new token and continue to use the query cells below.' % (info.access_token))
    return info	

tokenInfo = tokenInfoClass()

tokenInfo.email = input('Enter your XBRL US Web account email: ')
tokenInfo.password = getpass.getpass(prompt='Password: ')
tokenInfo.client_id = getpass.getpass(prompt='Client ID: ')
tokenInfo.client_secret = getpass.getpass(prompt='Secret: ')

body_auth = {'username' : tokenInfo.email, 
            'client_id': tokenInfo.client_id, 
            'client_secret' : tokenInfo.client_secret, 
            'password' : tokenInfo.password, 
            'grant_type' : 'password', 
            'platform' : 'ipynb' }

#print(body_auth)

payload = urlencode(body_auth)
res = requests.request("POST", tokenInfo.authurl, data=payload, headers=tokenInfo.headers)
auth_json = res.json()

if 'error' in auth_json:
    print("\n\nThere was a problem generating the access token: %s.  Run the first cell again and enter the credentials." % (auth_json['error_description']))
else:
    tokenInfo.access_token = auth_json['access_token']
    tokenInfo.refresh_token = auth_json['refresh_token']
    print ("\n\nYour access token expires in 60 minutes. After it expires, it should be regenerated automatically.  If not, run the cell rerun the first query cell. \n\nFor now, skip ahead to the section 'Make a Query'.")
	
#print(vars(tokenInfo))
print('\n\naccess token: ' + tokenInfo.access_token + ' refresh token: ' + tokenInfo.refresh_token)

### Make a query 
After the access token confirmation appears above, you can modify the code below to update the query, then run the cell to save it. In the next cell click run to query for results.
  
Refer to XBRL API documentation at https://xbrlus.github.io/xbrl-api/#/assertion/getAssertionDetails for other endpoints and parameters to filter and return. 

In [ ]:
# Define the parameters of the query - this query returns DQC assertions for specified years

endpoint = 'assertion'
XBRL_Elements = [
    'DQC.US.0094.9526','DQC.US.0094.9527','DQC.US.0107.9557','DQC.US.0107.9558','DQC.US.0110.9559','DQC.US.0113.9562','DQC.US.0114.9563','DQC.US.0116.9573','DQC.US.0117.9574','DQC.US.0120.9578','DQC.US.0122.9582','DQC.US.0124.9584','DQC.US.0124.9585','DQC.US.0124.9586','DQC.US.0124.9587','DQC.US.0124.9588','DQC.US.0125.9589','DQC.US.0116.9726','DQC.US.0116.9727','DQC.US.0131.9729','DQC.US.0132.9730','DQC.US.0139.9855','DQC.US.0139.9856','DQC.US.0139.9857','DQC.US.0139.9858','DQC.US.0139.9859','DQC.US.0139.9860','DQC.US.0140.9861','DQC.US.0140.9862','DQC.US.0142.9864','DQC.US.0143.9865','DQC.US.0144.9866','DQC.US.0145.9867','DQC.US.0145.9868','DQC.US.0145.9869','DQC.US.0146.9870','DQC.US.0147.9871','DQC.US.0148.9874','DQC.US.0149.9943','DQC.US.0150.9876','DQC.US.0150.9877','DQC.US.0150.9878','DQC.US.0153.9879','DQC.US.0154.10068','DQC.US.0154.10069','DQC.US.0154.10070','DQC.US.0154.10071','DQC.US.0155.10072','DQC.US.0148.10073','DQC.US.0156.10074','DQC.US.0149.10075','DQC.US.0150.10076','DQC.US.0157.10077','DQC.US.0159.10081','DQC.US.0160.10082','DQC.US.0161.10083','DQC.US.0162.10084','DQC.US.0163.10085','DQC.US.0164.10086','DQC.US.0164.10087','DQC.US.0164.10088','DQC.US.0164.10089','DQC.US.0164.10090','DQC.US.0165.10091','DQC.US.0160.10092','DQC.US.0117.10093','DQC.US.0174.10115','DQC.US.0174.10117'
    ]
report_year = [
    '2026',
    '2025',
    '2024'
    ]
fields = [ 
     # this is the list of the characteristics of the data being returned by the query
    #'report.entry-url',
    'report.accepted-timestamp.sort(DESC)',
    'assertion.code.sort(ASC)',
    'report.base-taxonomy.sort(DESC)',
    'report.document-type',
    'assertion.run-date',
    'report.accession',
    #'entity.code',
    #'entity.name',
    #'assertion.type',
    #'assertion.detail',
    'assertion.limit()'
    ]

# Set unique rows as True of False (True drops any duplicate rows)
unique = False

# Limit the number of rows displayed by the notebook (does not impact the data frame)
rows_to_display = 2 # Set as '' to display all rows in the notebook

# Below is the list of what's being queried using the search endpoint.
 
params = { 
    'assertion.code': ','.join(XBRL_Elements), 
    'report.filing-year': ','.join(report_year),
    'fields': ','.join(fields)
    }

print('\n\nclick the run button below to execute this query')

In [3]:
# @title
# ### Execute the query with loop for all results 
### THIS SECTION DOES NOT NEED TO BE EDITED

search_endpoint = baseurl + 'api/v1/' + endpoint + '/search'
if unique:
    search_endpoint += "?unique"
orig_fields = params['fields']
offset_value = 0
res_df = []
count = 0
query_start = datetime.now()
printed = False
run_query = True

while True:
    if not printed:
        print("On", query_start.strftime("%c"), tokenInfo.email, "(client ID:", str(tokenInfo.client_id.split('-')[0]), "...) started the query and")
        printed = True
    retry = 0
    while retry < 3:
        res = requests.get(search_endpoint, params=params, headers={'Authorization' : 'Bearer {}'.format(tokenInfo.access_token)})
        res_json = res.json()
        if 'error' in res_json:
            if res_json['error_description'] == 'Bad or expired token':
                tokenInfo = refresh(tokenInfo)
            else: 
                print('There was an error: {}'.format(res_json['error_description']))
                run_query = False
                break
        else: 
		        break
        retry +=1
        if retry >= 3:
            print("Can't refresh the access token.  Run the first query block, then rerun the query.")
            run_query = False

    if not run_query:
       break

    print("up to", str(offset_value + res_json['paging']['limit']), "records are found so far ...")

    res_df += res_json['data']

    if res_json['paging']['count'] < res_json['paging']['limit']:
        print(" - this set contained fewer than the", res_json['paging']['limit'], "possible, only", str(res_json['paging']['count']), "records.")
        break
    else: 
        offset_value += res_json['paging']['limit'] 
        if 100 == res_json['paging']['limit']:
                params['fields'] = orig_fields + ',' + endpoint + '.offset({})'.format(offset_value)
                if offset_value == 10 * res_json['paging']['limit']:
                        break 
        elif 500 == res_json['paging']['limit']:
                params['fields'] = orig_fields + ',' + endpoint + '.offset({})'.format(offset_value)
                if offset_value == 4 * res_json['paging']['limit']:
                        break 
        params['fields'] = orig_fields + ',' + endpoint + '.offset({})'.format(offset_value)

if not 'error' in res_json:
    current_datetime = datetime.now().replace(microsecond=0)
    time_taken = current_datetime - query_start
    index = pd.DataFrame(res_df).index
    total_rows = len(index)
    your_limit = res_json['paging']['limit']
    limit_message = "If the results below match the limit noted above, you might not be seeing all rows, and should consider upgrading (https://xbrl.us/access-token).\n"
    
    if your_limit == 100:
        print("\nThis non-Member account has a limit of " , 10 * your_limit, " rows per query from our Public Filings Database. " + limit_message)
    elif your_limit == 500:
        print("\nThis Basic Individual Member account has a limit of ", 4 * your_limit, " rows per query from our Public Filings Database. " + limit_message)
    
    print("\nAt " + current_datetime.strftime("%c") +  ", the query finished with  ", str(total_rows), "  rows returned in " + str(time_taken) + " for \n" +  urllib.parse.unquote(res.url))
    
    df = pd.DataFrame(res_df)
    # the format truncates the HTML display of numerical values to two decimals; .csv data is unaffected
    pd.options.display.float_format = '{:,.2f}'.format
    display(HTML(df.to_html(max_rows=rows_to_display)))

On Tue May 13 12:30:51 2025 test.tauriello@xbrl.us (client ID: 69e1257c ...) started the query and
up to 5000 records are found so far ...
up to 10000 records are found so far ...
up to 15000 records are found so far ...
up to 20000 records are found so far ...
up to 25000 records are found so far ...
 - this set contained fewer than the 5000 possible, only 766 records.

At Tue May 13 12:30:55 2025, the query finished with   20766   rows returned in 0:00:03.112940 for 
https://api.xbrl.us/api/v1/assertion/search?assertion.code=DQC.US.0094.9526,DQC.US.0094.9527,DQC.US.0107.9557,DQC.US.0107.9558,DQC.US.0110.9559,DQC.US.0113.9562,DQC.US.0114.9563,DQC.US.0116.9573,DQC.US.0117.9574,DQC.US.0120.9578,DQC.US.0122.9582,DQC.US.0124.9584,DQC.US.0124.9585,DQC.US.0124.9586,DQC.US.0124.9587,DQC.US.0124.9588,DQC.US.0125.9589,DQC.US.0116.9726,DQC.US.0116.9727,DQC.US.0131.9729,DQC.US.0132.9730,DQC.US.0139.9855,DQC.US.0139.9856,DQC.US.0139.9857,DQC.US.0139.9858,DQC.US.0139.9859,DQC.US.0139.9860,DQC.US.0

,report.accepted-timestamp,assertion.code,report.base-taxonomy,report.document-type,assertion.run-date,report.accession
0,2025-05-12 15:12:00,DQC.US.0140.9861,US GAAP 2024,10-Q,2025-05-13,0001411342-25-000052
...,...,...,...,...,...,...
20765,2024-01-04 16:30:00,DQC.US.0145.9867,US GAAP 2023,10-Q,2024-01-08,0001437749-24-000605


The code below filters the dataframe by beginning and ending times for SEC filings, then summarizes the dataframe by consolidating assertion code by major rule and adding a column summarizing the number of unique filings for each assertion code. 

In [4]:
# Convert 'report.accepted-timestamp' to datetime if it's not already
df['report.accepted-timestamp'] = pd.to_datetime(df['report.accepted-timestamp'])

# Define the date range
start_date = '2025-01-01 00:00:00'
end_date = '2025-04-01 00:00:00'

# Filter the dataframe
filtered_df = df[(df['report.accepted-timestamp'] >= start_date) & (df['report.accepted-timestamp'] < end_date)]

# Remove all characters in assertion.code after the third '.' (including the third '.')
filtered_df = filtered_df.map(lambda x: '.'.join(x.split('.')[:3]) if isinstance(x, str) else x)

# Count the occurrences of each assertion.code
assertion_code_counts = filtered_df.groupby('assertion.code').agg(
    count=('assertion.code', 'size'),
    filings=('report.accession', 'nunique')
).reset_index()

# Rename the columns for better readability
assertion_code_counts.columns = ['assertion.code', 'count', 'filings']

# Sort the assertion_code_counts table by 'count' in descending order
assertion_code_counts = assertion_code_counts.sort_values(by='count', ascending=False)

# Display the table
display(HTML(assertion_code_counts.to_html(index=False)))

# Display the updated dataframe
display(HTML(filtered_df.to_html(max_rows=rows_to_display)))

assertion.code,count,filings
DQC.US.0156,368,48
DQC.US.0161,189,24
DQC.US.0145,157,87
DQC.US.0150,151,24
DQC.US.0124,77,23
DQC.US.0164,52,10
DQC.US.0149,45,11
DQC.US.0116,37,8
DQC.US.0139,37,3
DQC.US.0163,27,3


,report.accepted-timestamp,assertion.code,report.base-taxonomy,report.document-type,assertion.run-date,report.accession
863,2025-03-31 20:57:00,DQC.US.0094,US GAAP 2024,10-K,2025-04-02,0001477932-25-002281
...,...,...,...,...,...,...
2184,2025-01-02 14:09:00,DQC.US.0156,US GAAP 2024,10-Q/A,2025-01-12,0001096906-25-000004


The cell below will save the initial dataframe to a local file or Google Drive as a .csv

In [ ]:
# If you run this program locally, you can save the output to a file 
# on your computer (modify D:\results.csv to your system)

df.to_csv(r"D:\assertions-public-exposure.csv",sep=",")

# Google Colab users - comment out the line above and uncomment the code below to save the data frame as a .csv in your Google Drive

#from google.colab import drive
#drive.mount('drive')
#df.to_csv('assertions-public-exposure.csv')
#!cp data.csv "drive/My Drive/"

The cell below will report the creation software used for each assertion in the dataframe.

In [ ]:
for accession in df['report.accession'].unique():
  url_report = f"https://api.xbrl.us/api/v1/report/search?report.accession={accession}&fields=report.accession,report.creation-software"
  response_report = requests.get(url_report, headers={'Authorization': 'Bearer {}'.format(tokenInfo.access_token)})
  accession_output = []
  if response_report.status_code == 200:
    report_data = response_report.json()
    #accession_output += report_data['data'][0]
    print(report_data['data'][0])

  elif response_report.status_code == 401:  # Unauthorized, token might have expired
      print("Authorization token expired. Try refreshing it.")
      tokenInfo = refresh(tokenInfo)  # Call your refresh function
  else:
    print(f"Error fetching data for {accession}: Status code {response_report.status_code}, {response_report.text}")

#print(accession_output)